# Krisk quick start

> **Krisk 0.9 notebook mode**
>
> This notebook runs locally with pandas and Krisk only. It does not require the Krisk
> server, MCP, SQLite, or PostgreSQL. Chart output embeds its renderer and can be used
> offline. The typed `ChartSpec` API is recommended for new work; legacy `krisk.plot`
> calls remain supported for existing notebooks.

Krisk creates interactive Apache ECharts visualizations from pandas-compatible data.
Start with the typed API when a chart may later be saved through REST, MCP, or living
research: the same validated specification works in every surface.

In [ ]:
from pathlib import Path

import pandas as pd
import krisk.plot as kk
from krisk import Chart, ChartSpec, __version__

ROOT = Path.cwd()
if not (ROOT / "krisk" / "tests" / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "krisk" / "tests" / "data"

print(f"Krisk {__version__}")

## Load repository-local example data

In [ ]:
gapminder = pd.read_csv(DATA_DIR / "gapminderDataFiveYear.txt", sep="\t")
gapminder.head()

## Recommended typed charts

In [ ]:
mean_life = (
    gapminder.groupby(["year", "continent"], as_index=False)["lifeExp"]
    .mean()
)

life_chart = Chart.from_dataframe(
    mean_life,
    ChartSpec(
        kind="line",
        x="year",
        y="lifeExp",
        color="continent",
        aggregate="mean",
        title="Life expectancy by continent",
        description="Five-year Gapminder observations, averaged by continent",
        smooth=True,
        height=480,
    ),
)
life_chart

Typed charts export to one self-contained HTML file. No server or CDN is needed.

In [ ]:
output = ROOT / "var" / "krisk" / "notebook-life-expectancy.html"
output.parent.mkdir(parents=True, exist_ok=True)
life_chart.to_html(output)
output

## Legacy plotting helpers

In [ ]:
kk.bar(gapminder, "continent")

Aggregate a measure and group it by color.

In [ ]:
kk.bar(gapminder, "year", y="gdpPercap", c="continent", how="mean")

Stack and annotate the same result.

In [ ]:
(
    kk.bar(
        gapminder,
        "year",
        y="gdpPercap",
        c="continent",
        how="mean",
        stacked=True,
        annotate=True,
    )
    .set_size(width=1000)
)

Histograms and themes remain available in the compatibility API.

In [ ]:
histogram = kk.hist(
    gapminder,
    x="lifeExp",
    c="continent",
    stacked=True,
    bins=30,
    density=True,
)
histogram.set_tooltip_style(trigger="axis", axis_pointer="shadow")
histogram.set_theme("vintage")

Scatter charts can encode size and color and use safe tooltip templates.

In [ ]:
scatter = kk.scatter(
    gapminder[gapminder.year == 2007],
    "lifeExp",
    "gdpPercap",
    s="pop",
    c="continent",
)
scatter.set_size(width=1000, height=500)
scatter.set_tooltip_format(["country", "lifeExp", "gdpPercap", "pop", "continent"])
scatter.set_theme("dark")
scatter.set_toolbox(save_format="png", restore=True, data_zoom=True)
scatter.set_legend(orient="vertical", x_pos="-1%", y_pos="-3%")
scatter.set_title("Gapminder in 2007", x_pos="center", y_pos="-5%")

## Where to go next

- Use `ChartSpec` for portable, validated chart definitions.
- Use `krisk.plot` when maintaining a legacy notebook.
- Start `uv run krisk serve` only when you want MCP, saved research, or Live tabs.
- Follow `docs/tutorial.md` for the complete conversation-to-research workflow.